# Hexgate policy DSL — complex-syntax demo

Exercises every new constraint construct and shows the **pydantic** engine (dev / fallback)
and the compiled **WASM** engine (signed production bundle) decide *identically*.

Constructs: `count()`, cross-field comparison, `every`/`any` quantifiers,
`startswith`/`endswith`/`contains`/`matches`, `and`/`or`/`not` + grouping,
named `consts.*`, and the `role`/`tool` facts — composed across roles + inheritance.

> Requires `opa` on PATH for the WASM cells (`brew install opa`).

In [ ]:
import shutil, yaml
from hexgate.security import compile_to_wasm, WasmPolicy, load_policy_set_from_dict
from hexgate.security.rego import compile_to_rego

assert shutil.which("opa"), "install opa: brew install opa"
print("opa found — WASM path enabled")

## The policy

One document, three roles (`base` mixin + `default` + `admin`) with rich per-tool rules.

In [ ]:
POLICY = """version: 1
roles:
  base:                                   # shared constants live in a mixin (2f)
    is_mixin: true
    consts:
      max_recipients: 5
      prod_env: "production"

  default:
    inherits: [base]
    tools:
      send_email:
        mode: allow
        constraints:
          - count(args.to) <= consts.max_recipients          # count() + named const
          - every(args.to, endswith(., "@acme.com"))         # quantifier over a list
      refund:
        mode: allow
        constraints:
          - args.amount <= args.limit                        # cross-field comparison
          - matches(args.ticket, "^INC-[0-9]+$")             # regex (RE2)
      read_file:
        mode: allow
        constraints:
          - startswith(args.path, "/srv/") and not contains(args.path, "..")  # boolean
      deploy:
        mode: approval_required
        constraints:
          - args.env != consts.prod_env or role == "admin"   # or + role fact

  admin:
    inherits: [base]
    tools:
      deploy: { mode: allow, constraints: ['role == "admin"'] }
"""
payload = yaml.safe_load(POLICY)
ps = load_policy_set_from_dict(payload)                       # pydantic engine
wasm = WasmPolicy.from_bytes(compile_to_wasm(compile_to_rego(payload)).wasm)  # WASM engine
print("roles:", ps.roles)

## Evaluate real tool calls through both engines

Each row is a `(role, tool, args)` decision. The **agree** column is the point: dev
(pydantic) and prod (WASM) must never disagree.

In [ ]:
def decide(role, tool, args):
    py = ps.evaluate(role=role, tool=tool, args=args).outcome.value
    d = wasm.decide(role=role, tool=tool, args=args)
    wo = 'allow' if d.allow else ('needs_approval' if d.requires_approval else 'deny')
    return py, wo

CASES = [
    ('default','send_email',{'to':['a@acme.com','b@acme.com']}),
    ('default','send_email',{'to':['x@gmail.com']}),                # not @acme.com
    ('default','send_email',{'to':['a@acme.com']*6}),               # > max_recipients
    ('default','refund',{'amount':50,'limit':100,'ticket':'INC-42'}),
    ('default','refund',{'amount':200,'limit':100,'ticket':'INC-42'}),   # amount>limit
    ('default','refund',{'amount':50,'limit':100,'ticket':'nope'}),      # bad ticket
    ('default','read_file',{'path':'/srv/app.log'}),
    ('default','read_file',{'path':'/srv/../etc/passwd'}),          # contains ..
    ('default','read_file',{'path':'/etc/passwd'}),                 # not /srv/
    ('default','deploy',{'env':'staging'}),                         # needs_approval
    ('default','deploy',{'env':'production'}),                      # deny (not admin)
    ('admin','deploy',{'env':'production'}),                        # admin -> allow
]
print(f"{'role':8} {'tool':11} {'pydantic':14} {'wasm':14} agree")
for role, tool, args in CASES:
    py, wo = decide(role, tool, args)
    print(f'{role:8} {tool:11} {py:14} {wo:14} {"OK" if py==wo else "MISMATCH!!"}')

## Edge cases: fail-closed + type safety

Missing args, wrong-typed args, empty collections — the cases the generative fuzzer
surfaced. Both engines must agree and stay safe (no fail-open).

In [ ]:
EDGE = [
    ('default','refund',{'limit':100,'ticket':'INC-1'}),                 # amount missing
    ('default','refund',{'amount':'lots','limit':100,'ticket':'INC-1'}), # wrong type
    ('default','send_email',{'to':[]}),                                  # every([]) -> allow
    ('default','send_email',{}),                                         # to missing
]
for role, tool, args in EDGE:
    py, wo = decide(role, tool, args)
    print(f'{tool:11} {str(args):52} py={py:6} wasm={wo:6} {"OK" if py==wo else "MISMATCH!!"}')

## Same policy, built in code (`PolicyBuilder` + `C`)

No YAML — typed constructors emit the same grammar, validated at the call site.

In [ ]:
from hexgate import PolicyBuilder, C, assert_allows, assert_denies

p = (PolicyBuilder(default='deny')
     .allow('refund', when=[
         C('args.amount') <= C('args.limit'),          # typed cross-field
         'matches(args.ticket, "^INC-[0-9]+$")',      # functions: raw grammar string
     ])
     .build())
assert_allows(p, 'refund', {'amount':10,'limit':100,'ticket':'INC-9'})
assert_denies(p, 'refund', {'amount':999,'limit':100,'ticket':'INC-9'})
print('PolicyBuilder + C: assertions passed')

## Full enforcement seam (`PolicyEnforcer` + `User`)

What framework adapters call: a tool invocation is gated by `decide()`, with the
caller's role resolved from the active `User` scope.

In [ ]:
from hexgate.runtime import User
from hexgate.security.enforcer import PolicyEnforcer

enforcer = PolicyEnforcer(ps, agent_name='demo')
with User(user_id='alice', role='admin').sync_scope():
    print('admin  deploy prod ->', enforcer.decide('deploy', {'env':'production'}).outcome.value)
with User(user_id='bob', role='default').sync_scope():
    d = enforcer.decide('deploy', {'env':'production'})
    print('default deploy prod ->', d.outcome.value)